# cleanning_old

Responsabilite:
- conserver une ancienne version du flux de nettoyage maritime
- servir de reference historique pour le pipeline actuel

Entrees:
- ancien classeur source de nettoyage

Sorties:
- anciennes bases nettoyees et essais de transformation

Execution:
- notebook d'archive, a executer uniquement pour consultation ou comparaison


In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Nettoyage de la consolidation des cahiers noirs
Supprime les lignes sans Horaire<br>
Converti Horaire en Date et horaire<br>
Garde Horaire, sera utile pour déterminer s'il fait jour (date + heure)<br>
Décompose la colonne météo<br>
Supprime les lignes sans Vent ou Houle<br>
réorganise les colonnes<br>
modifie la colonne cible : AnnulationMotif et Annulation (0 ou 1)<br>
Exporte maritime_clean.csv<br>

## Suite
1. Récupérer l'export créé ci-dessous avec le notebook **ligne_selector** pour selectionner la ligne de navette à étudier.
2. Et dans un second temps importer le fichier exporté depuis **ligne_selector** dans le notebook **maritime_processing**


In [19]:
source =  '../data/consolidation_maritime.xlsx'
df = pd.read_excel(
    source,
    header=0,
    skiprows=[1],
    usecols=['Horaire', 'Ligne', 'Annulation', 'Météo', 'Bateau', 'Capitaine']
    )

In [20]:
# Vérifie et supprime les ligne qui n'ont pas de 'Horaire'
# df['Horaire'].isnull().sum()
df.dropna(subset=['Horaire'], inplace=True)

# Convertit 'Horaire' au format datetime
df['Horaire'] = pd.to_datetime(df['Horaire'], format="%d/%m/%Y %H:%M", errors='coerce')
# Vérifie si 'Horaire' est bien de type datetime :
# pd.api.types.is_datetime64_any_dtype(df['Horaire'])

# Colonnes 'date' et 'heure'
df['Date'] = df['Horaire'].dt.date
df['Heure'] = df['Horaire'].dt.time

In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 63619 entries, 0 to 63666
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Horaire     63619 non-null  datetime64[ns]
 1   Ligne       63619 non-null  object        
 2   Annulation  10766 non-null  object        
 3   Météo       63619 non-null  object        
 4   Bateau      63619 non-null  object        
 5   Capitaine   63619 non-null  object        
 6   Date        63619 non-null  object        
 7   Heure       63619 non-null  object        
dtypes: datetime64[ns](1), object(7)
memory usage: 4.4+ MB


In [27]:
#  11163 lignes ont 'Météo' = ?
df['Météo'].value_counts().head(1)

#  'Météo', remplace les ? par None
# df.loc[df['Météo'] == '?', 'Météo'] = None

# Supprime les lignes 'Météo' = '?'
df = df[df['Météo'] != '?']

In [29]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 52456 entries, 0 to 63666
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Horaire     52456 non-null  datetime64[ns]
 1   Ligne       52456 non-null  object        
 2   Annulation  8999 non-null   object        
 3   Météo       52456 non-null  object        
 4   Bateau      52456 non-null  object        
 5   Capitaine   52456 non-null  object        
 6   Date        52456 non-null  object        
 7   Heure       52456 non-null  object        
dtypes: datetime64[ns](1), object(7)
memory usage: 3.6+ MB


### Décompose la colonne Météo en 3 parties :
- **Meteo1** = température + ciel (jusqu’à "Vent" ou "Mer" s’il n’y a pas de "Vent")
- **MeteoVent** = "Vent" → juste avant "Mer"
- **MeteoHoule** = "Mer" jusqu’à la fin

In [33]:
df['Meteo1'] = df['Météo'].str.extract(r'^(.*?)\b(Vent|Mer)\b', expand=False)[0]
# MeteoVent = de "Vent" jusqu’à "Mer", inclus
df['MeteoVent'] = df['Météo'].str.extract(r'(Vent.*?)(?=Mer\b)', expand=False)
df['MeteoHoule'] = df['Météo'].str.extract(r'(Mer\s*:?.*)$', expand=False)

In [35]:
# Meteo1
df['Temperature'] = df['Meteo1'].str.extract(r'(?P<Temperature>\d+)°')
df['Ciel'] = df['Meteo1'].str.extract(r'\d+°\s*(.*)')


In [37]:
# 'MeteoVent'
vent_regex = (
    r'Vent\s*:?\s*'                      # "Vent" (avec ou sans :)
    r'(?P<Vent>[A-Z]{1,3})?\s*'          # Direction OPTIONNELLE, si vent = 0 -> pas de direction
    r'(?P<VentNoeud>\d+)\s*Nds/?'        # Nœuds
    r'(?P<VentBeaufort>\d+)\s*Bft'       # Bft
)
vent_parsed = df['MeteoVent'].str.extract(vent_regex)
df = pd.concat([df, vent_parsed], axis=1)

In [39]:
#  corrige les collages entre mots et chiffres
df['MeteoHoule_clean'] = df['MeteoHoule'].str.replace(r'([a-zé])(\d)', r'\1 \2', regex=True)

In [41]:
# MeteoHoule
mer_regex = (
    r'Mer\s*:?\s*'                             # "Mer" avec ou sans ":"
    r'(?P<Mer>[^\d]+)?\s*'                     # Description (ex: "peu agitée")
    r'(?P<HouleDominante>\d+[.,]?\d*)/?'       # Houle dominante
    r'(?P<HouleMax>\d+[.,]?\d*)?m?\s*'         # Houle max
    r'(?P<Houle>\d+)?°?\s*'                    # Direction
    r'(?P<HoulePeriode>\d+)?s?'                # Période
)
mer_parsed = df['MeteoHoule'].str.extract(mer_regex)
df = pd.concat([df, mer_parsed], axis=1)

In [43]:
# Colonnes numériques : , deviens .
cols_to_float = ['Temperature', 'VentNoeud', 'VentBeaufort', 'HouleDominante', 'HouleMax', 'Houle', 'HoulePeriode']
for col in cols_to_float:
    df[col] = df[col].str.replace(',', '.').astype(float)

# supprime les colonnes redondantes et intermédiaires
df.drop(['Météo', 'Meteo1', 'MeteoVent', 'MeteoHoule'], axis=1, inplace = True)

In [45]:
# Vérifie et supprime les ligne qui n'ont pas de 'Vent' ou de 'Houle'
df.dropna(subset=['Vent','Houle'], inplace=True)

In [47]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 52421 entries, 0 to 63666
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Horaire           52421 non-null  datetime64[ns]
 1   Ligne             52421 non-null  object        
 2   Annulation        8993 non-null   object        
 3   Bateau            52421 non-null  object        
 4   Capitaine         52421 non-null  object        
 5   Date              52421 non-null  object        
 6   Heure             52421 non-null  object        
 7   Temperature       52421 non-null  float64       
 8   Ciel              52421 non-null  object        
 9   Vent              52421 non-null  object        
 10  VentNoeud         52421 non-null  float64       
 11  VentBeaufort      52421 non-null  float64       
 12  MeteoHoule_clean  52421 non-null  object        
 13  Mer               52421 non-null  object        
 14  HouleDominante    52421 non

In [49]:
df.head(2)

,Horaire,Ligne,Annulation,Bateau,Capitaine,Date,Heure,Temperature,Ciel,Vent,VentNoeud,VentBeaufort,MeteoHoule_clean,Mer,HouleDominante,HouleMax,Houle,HoulePeriode
0,2023-01-01 06:30:00,Vieux Port-Frioul,NaN,HJEsperandieu,Hellmann Nicolas,2023-01-01,06:30:00,14.0,Partiellement nuageux,E,10.0,3.0,"Mer : peu agitée 1/1,8m 122° 4s",peu agitée,1.0,1.8,122.0,4.0
1,2023-01-01 07:05:00,Frioul-Vieux Port,NaN,HJEsperandieu,Hellmann Nicolas,2023-01-01,07:05:00,14.0,Partiellement nuageux,E,8.0,3.0,"Mer : peu agitée 1,1/2m 132° 5s",peu agitée,1.1,2.0,132.0,5.0


### Annulation
- Renomme Annulation en  AnnulationMotif
-  création de la colonne Annulatio :
    - 1 = navette annulée 
    - 0 = la navette n'est pas annuléee

In [18]:
df.rename(columns={'Annulation': 'AnnulationMotif'}, inplace=True)

# Annulation en 0/1
df['Annulation'] = df['AnnulationMotif'].notna().astype(int)


In [19]:
colonnes_reorganisees = ['Horaire',
                         # 'Date',
                         # 'Heure',
                         'Annulation',
                         'AnnulationMotif',
                         'Ligne',
                         'Vent',
                         'VentNoeud',
                         # 'VentBeaufort',
                         'HouleDominante',
                         'HouleMax',
                         'Houle',
                         'HoulePeriode', 'Mer', 'Temperature', 'Ciel','Bateau', 'Capitaine']

df = df[colonnes_reorganisees]

In [20]:
# df.info()

In [21]:
# df['AnnulationMotif'].value_counts()

## Export dans le dossier data

In [23]:
df.to_csv("../data/maritime_clean.csv", index=True)

### Aller au notebook ligne_selector pour filtrer selon la ligne maritime

##### GARDE SEULEMENT LES COURSES ANNULEES

In [26]:
# df_annule = df[df['Annulation'] == True]
# df_annule.info()
# df_annule.describe()